In [0]:
/*
Static lookup tables to represent fiscal quarters.
*/
with dates as (
  select cast(m as date) as m, fq, fy, q
  , last_day(m) as last_day_of_month, year(m) as cy, month(m) as cm
  from (
    values 
      /*('2025-02-01', 'FY26-Q1', 2026, 1),
      ('2025-03-01', 'FY26-Q1', 2026, 1),
      ('2025-04-01', 'FY26-Q1', 2026, 1),
      ('2025-05-01', 'FY26-Q2', 2026, 2),
      ('2025-06-01', 'FY26-Q2', 2026, 2),
      ('2025-07-01', 'FY26-Q2', 2026, 2),
      ('2025-08-01', 'FY26-Q3', 2026, 3),
      ('2025-09-01', 'FY26-Q3', 2026, 3),
      ('2025-10-01', 'FY26-Q3', 2026, 3),
      ('2025-11-01', 'FY26-Q4', 2026, 4),
      ('2025-12-01', 'FY26-Q4', 2026, 4),
      ('2026-01-01', 'FY26-Q4', 2026, 4),*/
      ('2026-02-01', 'FY27-Q1', 2027, 1),
      ('2026-03-01', 'FY27-Q1', 2027, 1),
      ('2026-04-01', 'FY27-Q1', 2027, 1),
      ('2026-05-01', 'FY27-Q2', 2027, 2),
      ('2026-06-01', 'FY27-Q2', 2027, 2),
      ('2026-07-01', 'FY27-Q2', 2027, 2),
      ('2026-08-01', 'FY27-Q3', 2027, 3),
      ('2026-09-01', 'FY27-Q3', 2027, 3),
      ('2026-10-01', 'FY27-Q3', 2027, 3),
      ('2026-11-01', 'FY27-Q4', 2027, 4),
      ('2026-12-01', 'FY27-Q4', 2027, 4),
      ('2027-01-01', 'FY27-Q4', 2027, 4) 
  ) as dates(m, fq, fy, q)
),

-- Resolve :ae_email to a list of AE emails.
-- If :ae_email is an AE, returns just that email. If a manager, returns all AEs reporting to them.
ae_list as (
  select Email as ae_email, user_name, IsAE, level as sales_level
  from main.gtm_silver.individual_hierarchy_salesforce
  where snapshot_date = (select max(snapshot_date) from main.gtm_silver.individual_hierarchy_salesforce)
  --and IsAE = true
  and IsActive = true
  and Business_Unit = :business_unit
  and Region_Level_1 = :region_level_1
  and Region_Level_2 = :region_level_2
  and concatenated_emails like '%' || :ae_email || '%'
),

financial_quarters as (
  select fq, fy, q
    , max(last_day_of_month) as fiscal_quarter_end_date
    , min(m) as fiscal_quarter_start_date
    , case when getdate() > fiscal_quarter_end_date then q else null end as last_closed_q
    , case when getdate() > fiscal_quarter_end_date then true else false end as is_quarter_closed
    , sum(day(last_day_of_month)) as days_in_quarter
    , case when getdate() >= fiscal_quarter_start_date and current_date() <= fiscal_quarter_end_date then true else false end as is_current_fiscal_quarter
    , case when current_date() >= make_date(fy - 1, 2, 1) and current_date() <= make_date(fy, 1, 31) then 1 else 0 end as is_current_fiscal_year
    , (select max(usage_date) from main.gtm_gold.individual_consumption_daily)  as latest_usage_date
    , greatest(0, least(days_in_quarter, datediff(fiscal_quarter_end_date, latest_usage_date))) as days_left_in_quarter
    , case when is_current_fiscal_quarter then q else 0 end as current_quarter_number
  from dates
  group by fq, fy, q
),

targets as (
  SELECT ae.ae_email, t.Region_Level_1, t.Region_Level_2, t.Region_Level_3, t.user_id, t.Email, t.dollars as fin_target, t.fiscal_year, t.fiscal_quarter
  FROM gtm_silver.targets_individual t
  INNER JOIN ae_list ae ON t.Email = ae.ae_email
  where t.Business_Unit = :business_unit
  and t.Region_Level_1 = :region_level_1
  and t.Region_Level_2 = :region_level_2
  and t.type_target = 'dbu'
  and t.snapshot_date = (select max(snapshot_date) from gtm_silver.targets_individual where Region_Level_1 = :region_level_1 and Region_Level_2 = :region_level_2)
),

sales_forecast as (
  select ae.ae_email, f.forecast_owner_id as user_id, f.fiscal_quarter_end_date, f.submitted_my_call, f.submitted_my_call_w_closed_month_actuals
    , submitted_direct_field_consumption_forecast, current_ds_forecast_all_accounts, submitted_weighted_projection
  from gtm_silver.forecast_consumption_mcp_individual as f
  inner join ae_list ae on f.Email = ae.ae_email
  where f.Business_Unit = :business_unit
  and f.Region_Level_1 = :region_level_1
  and f.Region_Level_2 = :region_level_2
  and f.snapshot_date = (select max(snapshot_date) from gtm_silver.forecast_consumption_mcp_individual where Region_Level_1 = :region_level_1 and Region_Level_2 = :region_level_2)
),

actuals as (
  select ae.ae_email, c.fiscal_quarter_start_date, sum(c.dbu_dollars_qtd) as dbu_actuals
  , sum(c.dbu_dollars_t7d_avg) as dbu_dollars_t7d_avg, sum(c.dbu_dollars_t28d_avg) as dbu_dollars_t28d_avg
  , sum(c.dbu_dollars_t7d_avg_prev) as dbu_dollars_t7d_avg_prev, sum(c.dbu_dollars_t28d_avg_prev) as dbu_dollars_t28d_avg_prev
  from ae_list ae
  inner join main.gtm_gold.materialized__view_account_obt as c
    on c.concatenated_emails like '%' || ae.ae_email || '%'
  left outer join main.gtm_silver.account_dim as b
    on c.account_id = b.account_id
  where b.business_unit = :business_unit
  and b.region_level_1 = :region_level_1
  and b.region_level_2 = :region_level_2
  group by ae.ae_email, c.fiscal_quarter_start_date
),

target_forecast_actuals as (
  select d.fy, d.q, a.ae_email, a.fiscal_quarter_start_date, a.dbu_actuals
    , a.dbu_dollars_t7d_avg, a.dbu_dollars_t28d_avg, a.dbu_dollars_t7d_avg_prev, a.dbu_dollars_t28d_avg_prev
    , t.fin_target
    , f.submitted_my_call, f.submitted_direct_field_consumption_forecast, f.current_ds_forecast_all_accounts, f.submitted_weighted_projection
    , coalesce(a.dbu_actuals, 0) as dbu_actuals_coalesced
    , case when d.is_quarter_closed then a.dbu_actuals else submitted_my_call end dbu_actuals_or_forecast
    , first_value(a.dbu_dollars_t7d_avg) over(partition by a.ae_email order by d.is_current_fiscal_quarter desc) AS dbu_dollars_t7d_avg_latest
    , coalesce(a.dbu_dollars_t7d_avg, dbu_dollars_t7d_avg_latest) as dbu_dollars_t7d_adj --for future quarters, use the latest t7d available.
    , first_value(a.dbu_dollars_t28d_avg) over(partition by a.ae_email order by d.is_current_fiscal_quarter desc) AS dbu_dollars_t28d_avg_latest
    , coalesce(a.dbu_dollars_t28d_avg, dbu_dollars_t28d_avg_latest) as dbu_dollars_t28d_adj --for future quarters, use the latest t28d available.
    , first_value(a.dbu_dollars_t7d_avg_prev) over(partition by a.ae_email order by d.is_current_fiscal_quarter desc) AS dbu_dollars_t7d_avg_prev_latest
    , coalesce(a.dbu_dollars_t7d_avg_prev, dbu_dollars_t7d_avg_prev_latest) as dbu_dollars_t7d_prev_adj --for future quarters, use the latest t7d_prev available.
    , first_value(a.dbu_dollars_t28d_avg_prev) over(partition by a.ae_email order by d.is_current_fiscal_quarter desc) AS dbu_dollars_t28d_avg_prev_latest
    , coalesce(a.dbu_dollars_t28d_avg_prev, dbu_dollars_t28d_avg_prev_latest) as dbu_dollars_t28d_prev_adj --for future quarters, use the latest t28d_prev available.
    , case when d.is_quarter_closed then 0 else dbu_actuals_coalesced end dbu_actuals_current_quarter
    , case when d.is_quarter_closed then 0 else (dbu_dollars_t7d_adj * d.days_left_in_quarter) end as t7d_proj_left_in_quarter
    , case when d.is_quarter_closed then 0 else (dbu_dollars_t28d_adj * d.days_left_in_quarter) end as t28d_proj_left_in_quarter
    , try_divide(f.submitted_my_call - dbu_actuals_coalesced, d.days_left_in_quarter) as target_t7d  

  from actuals as a
  inner join financial_quarters d
  on d.fiscal_quarter_start_date = a.fiscal_quarter_start_date
  left outer join sales_forecast as f 
  on f.fiscal_quarter_end_date = d.fiscal_quarter_end_date and f.ae_email = a.ae_email
  left outer join targets as t 
  on t.fiscal_year = d.fy and t.fiscal_quarter = d.q and t.ae_email = a.ae_email
),

usecases_filtered as (
  select ae.ae_email, usecase_id, usecase_name, estimated_monthly_dollar_dbus, target_onboarding_date, target_live_date
    , dateadd(day, 14, date_trunc('month',target_onboarding_date)) as target_onboarding_date_15 
    , dateadd(day, 14, date_trunc('month',target_live_date)) as target_live_date_15
    , datediff(target_live_date, target_onboarding_date) as total_ramping_days 
    , date_format(dateadd(year, +1, dateadd(month, -1, target_onboarding_date)), "'FY'yy'-Q'Q") as target_onboarding_date_fq
    , date_format(dateadd(year, +1, dateadd(month, -1, target_live_date)), "'FY'yy'-Q'Q") as target_live_date_fq
    , concat('<a href="https://databricks.lightning.force.com/lightning/r/UseCase__c/', usecase_id, '/view" targe="_blank">',usecase_name,'</a>') as usecase_url
    , coalesce(num_of_blockers, 0) as num_of_blockers
    , case
        when days_in_stage <= 30 or days_in_stage is null then '0-30 days'
        when days_in_stage > 30 and days_in_stage <= 60 then '31-60 days'
        when days_in_stage > 60 and days_in_stage <= 120 then '61-120 days'
        when days_in_stage > 120 then '120+ days'
      end as days_in_stage_bucket
    , date_diff(DAY, current_date(), last_day(target_live_date)) as days_to_go_live
    , case when days_to_go_live < 0 then true else false end go_live_in_the_past
    , date_diff(DAY, current_date(), last_day(target_onboarding_date)) as days_to_onboarding

     --Check hygiene issues and risks
    , case
        when go_live_in_the_past then named_struct('category', 'Hygiene', 'msg', 'Go live date in the past')
        when implementation_status is null then named_struct('category', 'Hygiene', 'msg', 'Health status not defined')
        when days_to_onboarding < 0 and stage_number < 5 then named_struct('category', 'Hygiene', 'msg', 'Past Onboarding date / not U5') 
        when days_to_go_live < 30 and stage_number < 5 then named_struct('category', 'Warning', 'msg', 'Go live < 30 / Not U5')
        when days_to_go_live < 30 and implementation_status = 'Red' then named_struct('category', 'Warning', 'msg', 'Go live < 30 days / Red')
        when days_to_go_live < 30 and implementation_status = 'Yellow' then named_struct('category', 'Warning', 'msg', 'Go live < 30 days / Yellow')
        when days_to_go_live < 30 then named_struct('category', 'Warning', 'msg', 'Go live < 30')
        when days_to_onboarding between 0 and 30 and stage_number < 5 and implementation_status = 'Red' then named_struct('category', 'Warning', 'msg', 'Onboarding < 30 / Red')
        when days_to_onboarding between 0 and 30 and stage_number < 5 and implementation_status = 'Yellow' then named_struct('category', 'Warning', 'msg', 'Onboarding < 30 / Yellow')
        when days_to_onboarding between 0 and 30 and stage_number <= 3 then named_struct('category', 'Warning', 'msg', 'Onboarding < 30 / <=U3')
        when days_to_onboarding between 0 and 30 and stage_number = 4 then named_struct('category', 'Warning', 'msg', 'Onboarding < 30 / U4')
        when days_to_go_live < 60 and stage_number = 5 then named_struct('category', 'Opportunity', 'msg', 'Go live < 60 / U5')
        when days_to_onboarding between 0 and 60 and stage_number between 2 and 3 then named_struct('category', 'Opportunity', 'msg', 'Tech win to accelerate')
      end as uco_info
  , coalesce(implementation_status, 'Unknown') as implementation_status
    
  from gtm_silver.use_case_detail
  inner join ae_list ae on use_case_detail.concatenated_emails like '%' || ae.ae_email || '%'
  where Business_Unit = :business_unit
  and sales_subregion_level_1 = :region_level_1
  and sales_subregion_level_2 = :region_level_2
  and is_incremental = true --Excludes upgrades
  and stage_number <= 5 --Filter out 'Disqualified', 'Lost' and 'Live' UCOs.
  and coalesce(estimated_monthly_dollar_dbus, 0) > 0 -- Excludes zero-valued use cases.
  --and usecase_id = 'aAv8Y000000CsLuSAK' 
  /* test cases 
  aAv8Y000000CsLuSAK (Feb25->Sep25), aAvVp000000Uc6IKAS (May25->Sep25), aAv8Y000000lD0ySAE (Jun25->Dec25)
  aAvVp000000W9JVKA0 (Apr25->May25), aAvVp000000d2YEKAY (Feb25->Jul25), aAvVp000000WAqfKAG (Jun25->Jan26)
  */
),

incremental_projections as (
  select uco.ae_email, uco.usecase_id, d.fq, d.fy, d.q, d.m, d.cm, d.last_day_of_month
    , uco.target_onboarding_date, uco.target_onboarding_date_fq, uco.target_live_date, uco.target_live_date_fq, uco.target_onboarding_date_15, uco.target_live_date_15
    , uco.total_ramping_days, uco.estimated_monthly_dollar_dbus, uco.implementation_status, uco.usecase_url, uco.num_of_blockers 
    , datediff(getdate(), target_onboarding_date_15) as current_ramping_days 
    -- Calculate this month's baseline for each use case, .i.e. how much are they consuming today? This is used to calculate the actual incremental consumption at the next step.
    -- We assume that the onboarding date and live date occur on day 15 of the month.
    ,case when d.m between uco.target_onboarding_date and uco.target_live_date then 1 else 0 end as is_onboarding
  
    ,case         
        -- if UCO not onboarded yet (i.e. the onboarding date is in the future), then no dbus are generated for the current month.
        when target_onboarding_date_15 > getdate() then 0
        -- if UCO is already live, then it should already realise the expected monthly $DBUs.
        when getdate() > target_live_date_15 then estimated_monthly_dollar_dbus
        -- if UCO is currently onboarding (i.e. the onboarding date is in the past), this is the estimated dbus for the full current month. 
        else round(estimated_monthly_dollar_dbus * try_divide(datediff(getdate(), target_onboarding_date_15), total_ramping_days)) 
      end as current_dbu_baseline 

    --calculate the ramping dbus assuming a linear ramp between the tonboarding date and the go-live: from 0 $dbus to the expected monthly $dbus that will be reached on go-live.
    , case       
        when d.last_day_of_month < uco.target_onboarding_date_15 then 0 --Before the onboarding date
        when d.last_day_of_month > uco.target_live_date_15 then uco.estimated_monthly_dollar_dbus --After the go-live the $dbus remain flat 
        else round(uco.estimated_monthly_dollar_dbus * try_divide(datediff(d.last_day_of_month, uco.target_onboarding_date_15), uco.total_ramping_days)) --Between onboarding and go-live the dbus ramp-up linearly
      end as ramping_dbus

    --remove the realised dbus from the ramp, when the use case is ramping up during the onboarding phase, past months' revenue has already been realised.
    , case 
      when d.last_day_of_month < getdate() then 0 --Past month: if a use case started onboarding in the past, and the month is closed then we are removing the consumption from the pipeline to avoid double counting, because we assume it has already been realised (actual dbus).
      when d.last_day_of_month < uco.target_onboarding_date_15 then 0 --Before the onboarding date
      when d.last_day_of_month > uco.target_live_date_15 then uco.estimated_monthly_dollar_dbus - current_dbu_baseline --After the go-live
      else round(uco.estimated_monthly_dollar_dbus * try_divide(datediff(d.last_day_of_month, uco.target_onboarding_date_15), uco.total_ramping_days)) - current_dbu_baseline --Between onboarding and go-live 
    end as ramping_dbus_from_baseline 

     -- calculates the actual incremental value substracting last month's $dbus from this month's $dbus.
    , case 
        when getdate() > m then ramping_dbus - ramping_dbus_from_baseline
        else 0 --in the future
    end as dbus_generated

  from usecases_filtered as uco
  inner join dates as d --cross join with date table
),

quarterly_projection_by_use_case as (
  select              
    i.ae_email, i.usecase_id, i.fy, i.fq, i.q
    , sum(i.ramping_dbus) as quarterly_ramping_dbus    
    , sum(i.dbus_generated) as quarterly_dbus_generated
    , lag(max(i.ramping_dbus)) over (partition by i.ae_email, i.usecase_id order by i.fq asc) as last_day_of_prev_quarter_dbus
    from incremental_projections as i    
    group by all
),

monthly_projection as (
  select
    ip.ae_email, ip.usecase_id, ip.fy, ip.fq, ip.q, ip.m, ip.cm, ip.ramping_dbus, ip.current_dbu_baseline, ip.is_onboarding
    , f.last_closed_q, f.days_left_in_quarter, f.is_quarter_closed, f.is_current_fiscal_quarter
    , qp.last_day_of_prev_quarter_dbus    
    , greatest(qp.last_day_of_prev_quarter_dbus, ip.current_dbu_baseline) as quarter_dbu_baseline

    -- Incremental quarterly $dbus. 
    -- If the quarter has started then we use the current baseline to identify addtional incremental dbus until the end of the quarter
    -- if the quarter has not started yet the baseline is the last day of the previous quarter.    
    ,case 
        when ip.ramping_dbus - greatest(qp.last_day_of_prev_quarter_dbus, ip.current_dbu_baseline) < 0 then 0 -- All past months do not contribute to incremental dbus. 
        else ip.ramping_dbus - greatest(qp.last_day_of_prev_quarter_dbus, ip.current_dbu_baseline)
     end as quarterly_incremental_dbus

    , case when ip.implementation_status = 'Green' and not f.is_quarter_closed then quarterly_incremental_dbus else 0 end as dbus_in_pipeline_green 
    , case when ip.implementation_status = 'Yellow' and not f.is_quarter_closed then quarterly_incremental_dbus else 0 end as dbus_in_pipeline_yellow 
    , case when ip.implementation_status = 'Red' and not f.is_quarter_closed then quarterly_incremental_dbus else 0 end as dbus_in_pipeline_red
    , case when ip.implementation_status = 'Unknown' and not f.is_quarter_closed then quarterly_incremental_dbus else 0 end as dbus_in_pipeline_unknown
  
  from incremental_projections as ip
  inner join quarterly_projection_by_use_case as qp
  on ip.usecase_id = qp.usecase_id
  and ip.fq = qp.fq
  and ip.ae_email = qp.ae_email
  inner join financial_quarters as f
  on ip.fq = f.fq
),

quarterly_projection as (
  select p.ae_email, p.fy, p.fq, p.q, p.last_closed_q, p.days_left_in_quarter, p.is_quarter_closed, p.is_current_fiscal_quarter
    , sum(p.quarterly_incremental_dbus) as quarterly_incremental_dbus
    , sum(p.dbus_in_pipeline_green) as dbus_in_pipeline_green 
    , sum(p.dbus_in_pipeline_yellow) as dbus_in_pipeline_yellow 
    , sum(p.dbus_in_pipeline_red) as dbus_in_pipeline_red
    , sum(p.dbus_in_pipeline_unknown) as dbus_in_pipeline_unknown
    , sum(last_day_of_prev_quarter_dbus) as last_day_of_prev_quarter_dbus

    from monthly_projection as p
    group by all
),

quarterly_summary as (
  SELECT 
  p.ae_email, ae.sales_level, ae.user_name, p.fy, p.fq, p.days_left_in_quarter, p.is_quarter_closed , p.is_current_fiscal_quarter
  , f.fin_target, f.submitted_my_call 
  , f.submitted_direct_field_consumption_forecast, f.current_ds_forecast_all_accounts, f.submitted_weighted_projection
  , f.dbu_actuals_coalesced as dbu_actuals, f.dbu_actuals_or_forecast, f.dbu_actuals_current_quarter
  , f.dbu_dollars_t7d_adj, f.dbu_dollars_t7d_prev_adj, f.t7d_proj_left_in_quarter, f.target_t7d
  , f.dbu_dollars_t28d_adj, f.dbu_dollars_t28d_prev_adj, f.t28d_proj_left_in_quarter

  , coalesce(lag(f.dbu_actuals_or_forecast) over (partition by p.ae_email order by p.fq asc), 0) as dbu_actuals_or_forecast_prev_quarter
  , try_divide(f.fin_target - dbu_actuals_or_forecast_prev_quarter, dbu_actuals_or_forecast_prev_quarter) as qoq_target_growth

  --incremental pipeline
  , p.quarterly_incremental_dbus 
  , p.dbus_in_pipeline_green, p.dbus_in_pipeline_yellow, p.dbus_in_pipeline_red, p.dbus_in_pipeline_unknown
  , p.dbus_in_pipeline_green * :green_confidence_pct as dbus_in_pipeline_green_in_plan
  , p.dbus_in_pipeline_yellow * :yellow_confidence_pct as dbus_in_pipeline_yellow_in_plan
  , p.dbus_in_pipeline_red * :red_confidence_pct as dbus_in_pipeline_red_in_plan
  , p.dbus_in_pipeline_unknown * :unknown_confidence_pct as dbus_in_pipeline_unknown_in_plan

  --set the baseline: 
  --for the current quarter, use the T7D because more precise. 
  --for future quarters use 
  , case when p.is_quarter_closed then 0 when p.is_current_fiscal_quarter then f.t7d_proj_left_in_quarter else dbu_actuals_or_forecast_prev_quarter  end as baseline
  , case when p.is_quarter_closed then "Baseline" when p.is_current_fiscal_quarter then "Baseline (T7D Projection + OG)" else "Baseline (previous quarter's forecast) + OG" end as baseline_label -- used as a label in the dashboard

  , case when p.is_quarter_closed then 0 else f.dbu_actuals_coalesced + f.t7d_proj_left_in_quarter end as t7d_flat_projection 
  , case when p.is_quarter_closed then 0 else f.dbu_actuals_coalesced + f.t28d_proj_left_in_quarter end as t28d_flat_projection

  --best case
  , baseline * :best_case_qoq_organic_growth as best_case_proj_with_og_left_in_quarter
  , dbus_in_pipeline_green_in_plan + dbus_in_pipeline_yellow_in_plan + dbus_in_pipeline_red_in_plan + dbus_in_pipeline_unknown_in_plan as best_case_pipe_left_in_quarter
  , f.dbu_actuals_coalesced + best_case_proj_with_og_left_in_quarter + best_case_pipe_left_in_quarter + :best_case_adjustments as best_case_projection

  --worst case
  , baseline * :worst_case_qoq_organic_growth as worst_case_proj_with_og_left_in_quarter
  , dbus_in_pipeline_green_in_plan as worst_case_pipe_left_in_quarter
  , f.dbu_actuals_coalesced + worst_case_proj_with_og_left_in_quarter + worst_case_pipe_left_in_quarter + :worst_case_adjustments as worst_case_projection
  
  --forecast gaps
  , f.submitted_my_call - f.fin_target as gap_my_call
  , best_case_projection - f.fin_target as gap_best_case
  , worst_case_projection - f.fin_target as gap_worst_case
  , t7d_flat_projection - f.fin_target as gap_t7d_flat_projection
  , t28d_flat_projection - f.fin_target as gap_t28d_flat_projection
  , f.submitted_weighted_projection - f.fin_target as gap_weighted_projection
  , f.current_ds_forecast_all_accounts - f.fin_target as gap_ds_forecast
  , f.submitted_direct_field_consumption_forecast - f.fin_target as gap_directs_forecast

   --QoQ delta: used to calculate the QoQ % groeth
  , f.submitted_my_call - dbu_actuals_or_forecast_prev_quarter as qoq_delta_target 
  , best_case_projection - dbu_actuals_or_forecast_prev_quarter as qoq_delta_best_case
  , worst_case_projection - dbu_actuals_or_forecast_prev_quarter as qoq_delta_worst_case
  , t7d_flat_projection - dbu_actuals_or_forecast_prev_quarter as qoq_delta_t7d_flat_projection
  , t28d_flat_projection - dbu_actuals_or_forecast_prev_quarter as qoq_delta_t28d_flat_projection
  , f.submitted_weighted_projection - dbu_actuals_or_forecast_prev_quarter as qoq_delta_weighted_projection
  , f.current_ds_forecast_all_accounts - dbu_actuals_or_forecast_prev_quarter as qoq_delta_ds_forecast
  , f.submitted_direct_field_consumption_forecast - dbu_actuals_or_forecast_prev_quarter as qoq_delta_directs_forecast

  --labels
  , format_number(try_divide(f.submitted_my_call, f.fin_target), '#.#%') || 
    " | " || format_number(try_divide(qoq_delta_target, dbu_actuals_or_forecast_prev_quarter), '#.#%') || 
    " | " || format_number(gap_my_call, '$,###.#') as my_call_text
  , format_number(try_divide(f.submitted_direct_field_consumption_forecast, f.fin_target), '#.#%') ||
    " | " || format_number(try_divide(qoq_delta_directs_forecast, dbu_actuals_or_forecast_prev_quarter), '#.#%') ||
    " | " || format_number(gap_directs_forecast, '$,###.#') as directs_fct_text
  , format_number(try_divide(gap_best_case, f.fin_target), '#.#%') || 
    " | " || format_number(try_divide(qoq_delta_best_case, dbu_actuals_or_forecast_prev_quarter), '#.#%') || 
    " | " || format_number(gap_best_case, '$,###.#') as best_case_text
  , format_number(try_divide(worst_case_projection, f.fin_target), '#.#%') || 
    " | " || format_number(try_divide(qoq_delta_worst_case, dbu_actuals_or_forecast_prev_quarter), '#.#%') || 
    " | " || format_number(gap_worst_case, '$,###.#') as worst_case_text
  , format_number(try_divide(submitted_weighted_projection, f.fin_target), '#.#%') || 
    " | " || format_number(try_divide(qoq_delta_weighted_projection, dbu_actuals_or_forecast_prev_quarter), '#.#%') || 
    " | " || format_number(gap_weighted_projection, '$,###.#') as weighted_proj_text
  , format_number(try_divide(current_ds_forecast_all_accounts, f.fin_target), '#.#%') || 
    " | " || format_number(try_divide(qoq_delta_ds_forecast, dbu_actuals_or_forecast_prev_quarter), '#.#%') || 
    " | " || format_number(gap_ds_forecast, '$,###.#') as ds_forecast_text
  , format_number(try_divide(t7d_flat_projection, f.fin_target), '#.#%') || 
    " | " || format_number(try_divide(qoq_delta_t7d_flat_projection, dbu_actuals_or_forecast_prev_quarter), '#.#%') || 
    " | " || format_number(gap_t7d_flat_projection, '$,###.#') as t7d_proj_text
  , format_number(try_divide(t28d_flat_projection, f.fin_target), '#.#%') || 
    " | " || format_number(try_divide(qoq_delta_t28d_flat_projection, dbu_actuals_or_forecast_prev_quarter), '#.#%') || 
    " | " || format_number(gap_t28d_flat_projection, '$,###.#') as t28d_proj_text

  from quarterly_projection as p
  left outer join target_forecast_actuals as f 
  on f.fy = p.fy and f.q = p.q and f.ae_email = p.ae_email
  inner join ae_list ae on ae.ae_email = p.ae_email
)

/* Forecast waterfall for current user only */
select * from quarterly_summary 
--where ae_email = :ae_email

--Forecast waterfall for current user and all its AEs
/*
select fy, fq, ae_email, user_name,  sales_level, submitted_my_call, fin_target, gaps, value
from quarterly_summary
unpivot (
  value for gaps in (
    gap_to_target as `Gap to Target`,
    gap_to_ds_forecast as `Gap to DS`,
    gap_to_worst_case as `Gap to Worst Case`,
    gap_to_best_case as `Gap to Best Case`,
    gap_to_t7d_flat_projection as `Gap to T7D Proj`,
    gap_to_weighted_projection as `Gap to WP`
  )
)
*/